# Corpus exploration

Überblick über den geparsten PubMed-Korpus in `data/processed/`.

Inhalt:
1. Filter, mit denen die Roh-XMLs aus PMC OA gezogen wurden
2. Korpusgröße, Journal-Verteilung
3. Sektionsstruktur und häufigste Section-Titel
4. Längenverteilung (Body-Zeichen, Sections pro Paper)
5. Abstract-Strukturen
6. Beispiel-Paper im Detail

## Setup

In [40]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
RAW_DIR = REPO_ROOT / "data" / "raw"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)
print(f"processed: {PROCESSED_DIR}")
print(f"raw:       {RAW_DIR}")

processed: /Users/jan/Developer/ezmed/ezmed-rag-retrieval/data/processed
raw:       /Users/jan/Developer/ezmed/ezmed-rag-retrieval/data/raw


In [41]:
records = []
for path in sorted(PROCESSED_DIR.glob("*.json")):
    d = json.loads(path.read_text())
    records.append({
        "pmid": d["pmid"],
        "pmcid": d["pmcid"],
        "doi": d["doi"],
        "title": d["title"],
        "journal": d["journal"],
        "n_authors": len(d["authors"]),
        "n_abstract_secs": len(d["abstract"]),
        "n_body_secs": len(d["body"]),
        "abstract_chars": sum(len(s["text"]) for s in d["abstract"]),
        "body_chars": sum(len(s["text"]) for s in d["body"]),
        "body_section_titles": [s["title"] for s in d["body"]],
    })
df = pd.DataFrame(records)
print(f"loaded {len(df)} papers")
df.head(3)

loaded 1000 papers


,pmid,pmcid,doi,title,journal,n_authors,n_abstract_secs,n_body_secs,abstract_chars,body_chars,body_section_titles
0,36901372,10002348,10.3390/ijerph20054366,Organisational Impact of a Remote Patient Monitoring System for Heart Failur...,International Journal of Environmental Research and Public Health,12,1,5,1537,20679,"[1. Introduction, 2. Methods, 3. Results, 4. Discussion, 5. Conclusions]"
1,36525339,10011328,10.1093/eurheartj/ehac633,Randomized trials fit for the 21st century. A joint opinion from the Europea...,European Heart Journal,63,1,5,18,17358,"[Problem, Background, Opportunity for global impact, Addressing the challeng..."
2,36918867,10014396,10.1186/s12904-023-01138-z,Health care providers’ perspectives on providing end-of-life psychiatric car...,BMC Palliative Care,9,5,6,2516,19539,"[Background, Methods, Results, Discussion, Conclusion, Electronic supplement..."


## 1. Filter

Die Roh-XMLs in `data/raw/` stammen aus der PMC-Open-Access-Subset über die NCBI-E-Utilities. Die Suche in `src/ezmed/ingestion/pubmed.py:_build_query` ist ein einfaches MeSH-AND:

```
{domain}[MeSH] AND open access[filter] AND english[lang]
```

Für die Thesis ist `domain` auf Cardiology fixiert. Die Einschränkungen:

- **`open access[filter]`** — nur Volltexte, die wir kostenfrei und ohne Lizenzthemen verarbeiten dürfen.
- **`english[lang]`** — die Embedding-Modelle und LLM-Prompts laufen Englisch.
- **`{domain}[MeSH]`** — thematische Eingrenzung über MeSH-Indexterme statt Volltext-Match (geringere Recall-Verzerrung).

Die abgerufenen PMC-IDs sind in `data/raw/_pmcids.txt` festgehalten.

In [42]:
from ezmed.ingestion.pubmed import _build_query

print(_build_query("cardiology"))

pmcids_file = RAW_DIR / "_pmcids.txt"
if pmcids_file.exists():
    n_listed = sum(1 for _ in pmcids_file.read_text().splitlines() if _.strip())
    print(f"\n_pmcids.txt: {n_listed} IDs gelistet")
n_xml = sum(1 for _ in RAW_DIR.glob("*.xml"))
print(f"data/raw/*.xml: {n_xml} Dateien gecached")
print(f"data/processed/*.json: {len(df)} Dateien geparst")

cardiology[MeSH] AND open access[filter] AND english[lang]

_pmcids.txt: 1000 IDs gelistet
data/raw/*.xml: 1000 Dateien gecached
data/processed/*.json: 1000 Dateien geparst


## 2. Korpus-Größe und Journals

In [43]:
print(f"Papers:        {len(df)}")
print(f"Unique PMIDs:  {df['pmid'].nunique()}")
print(f"Mit DOI:       {df['doi'].notna().sum()}  ({df['doi'].notna().mean():.0%})")
print(f"Unique Journals: {df['journal'].nunique()}")
print(f"Ohne Journal:  {df['journal'].isna().sum()}")

Papers:        1000
Unique PMIDs:  1000
Mit DOI:       998  (100%)
Unique Journals: 278
Ohne Journal:  0


In [55]:
df["journal"].value_counts().head(15).to_frame("papers")

,papers
journal,
Journal of the American Heart Association: Cardiovascular and Cerebrovascular Disease,68
Arquivos Brasileiros de Cardiologia,56
Europace,36
Scientific Reports,32
International Journal of Molecular Sciences,21
ESC Heart Failure,21
Open Heart,21
European Heart Journal,20
Journal of the American College of Cardiology,20


## 3. Sektionsstruktur

Wieviele Body-Sections pro Paper, und welche Section-Titel kommen am häufigsten vor? Hilft beim Designen der Filter-Liste im Chunker (Introduction/Funding/etc. werden dort gedroppt, nicht im Parser).

In [45]:
df["n_body_secs"].describe().round(1).to_frame("body sections per paper")

,body sections per paper
count,1000.0
mean,5.8
std,3.8
min,0.0
25%,4.0
50%,5.0
75%,7.0
max,56.0


In [46]:
title_counter: Counter[str] = Counter()
for titles in df["body_section_titles"]:
    title_counter.update(t.strip().lower() for t in titles)

top_titles = pd.DataFrame(title_counter.most_common(30), columns=["section_title", "count"])
top_titles["share_of_papers"] = (top_titles["count"] / len(df)).round(2)
top_titles

,section_title,count,share_of_papers
0,introduction,538,0.54
1,discussion,412,0.41
2,results,396,0.40
3,methods,347,0.35
4,conclusions,230,0.23
5,conclusion,223,0.22
6,supplementary material,97,0.10
7,supporting information,87,0.09
8,disclosures,86,0.09
9,1. introduction,78,0.08


In [47]:
import re

FILTER_CANDIDATES = [
    "introduction", "background",
    "author contributions", "funding", "acknowledgments", "acknowledgements",
    "conflicts of interest", "conflict of interest",
    "data availability", "data availability statement",
    "supporting information", "supplementary material",
    "ethics statement",
]
_NUM_PREFIX = re.compile(r"^\d+(\.\d+)*\.?\s*")

def normalize(title: str) -> str:
    return _NUM_PREFIX.sub("", title.strip().lower())

rows = []
for cand in FILTER_CANDIDATES:
    n_papers = sum(
        any(normalize(t) == cand or normalize(t).startswith(cand) for t in titles)
        for titles in df["body_section_titles"]
    )
    rows.append({"candidate": cand, "papers_with": n_papers, "share": n_papers / len(df)})
filter_overview = pd.DataFrame(rows).sort_values("papers_with", ascending=False).reset_index(drop=True)
filter_overview["share"] = filter_overview["share"].round(2)
filter_overview

,candidate,papers_with,share
0,introduction,623,0.62
1,supporting information,87,0.09
2,supplementary material,87,0.09
3,background,65,0.06
4,conflict of interest,49,0.05
5,funding,44,0.04
6,author contributions,34,0.03
7,conflicts of interest,28,0.03
8,acknowledgments,8,0.01
9,acknowledgements,5,0.00


## 4. Längenverteilung

Body-Zeichen pro Paper informiert die Wahl der Chunk-Größe (aktuell `CHUNK_SIZE=1000`, `CHUNK_OVERLAP=200`).

In [48]:
df[["abstract_chars", "body_chars", "n_authors"]].describe().round(0)

,abstract_chars,body_chars,n_authors
count,1000.0,1000.0,1000.0
mean,1202.0,26283.0,13.0
std,807.0,29326.0,97.0
min,0.0,0.0,0.0
25%,449.0,14421.0,4.0
50%,1426.0,21958.0,7.0
75%,1780.0,30486.0,11.0
max,3366.0,451448.0,3021.0


In [49]:
quantiles = df["body_chars"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0)
est_chunks = (df["body_chars"] / 1000).round().astype(int)
print("Body-Zeichen-Quantile:")
print(quantiles.to_string())
print(f"\nGeschätzte Chunks insgesamt (Body-Zeichen / 1000): {est_chunks.sum():,}")
print(f"Ø Chunks pro Paper: {est_chunks.mean():.1f}")

Body-Zeichen-Quantile:
0.10       210.0
0.25     14421.0
0.50     21958.0
0.75     30486.0
0.90     45132.0
0.99    140781.0

Geschätzte Chunks insgesamt (Body-Zeichen / 1000): 26,274
Ø Chunks pro Paper: 26.3


## 5. Abstract-Strukturen

JATS-Abstracts sind entweder *strukturiert* (mehrere `<sec>` mit Titeln wie Background/Methods/Results/Conclusions) oder *flach* (nur `<p>`-Absätze, dann liefert der Parser eine einzelne Section mit `title="Abstract"`).

In [50]:
abstract_dist = df["n_abstract_secs"].value_counts().sort_index().to_frame("papers")
abstract_dist["share"] = (abstract_dist["papers"] / len(df)).round(2)
abstract_dist

,papers,share
n_abstract_secs,,
0,209,0.21
1,425,0.42
2,4,0.00
3,101,0.10
4,141,0.14
5,98,0.10
6,13,0.01
7,3,0.00
8,4,0.00


## 6. Beispiel-Paper

Ein längeres und ein kürzeres Paper, jeweils mit Section-Aufbau.

In [51]:
def show_paper(pmcid: str) -> None:
    path = PROCESSED_DIR / f"{pmcid}.json"
    d = json.loads(path.read_text())
    print(f"PMCID:   {d['pmcid']}")
    print(f"PMID:    {d['pmid']}")
    print(f"DOI:     {d['doi']}")
    print(f"Journal: {d['journal']}")
    print(f"Title:   {d['title']}")
    print(f"Authors: {len(d['authors'])} (first 3: {', '.join(d['authors'][:3])})")
    print(f"\nAbstract sections ({len(d['abstract'])}):")
    for s in d["abstract"]:
        print(f"  - {s['title']}  ({len(s['text'])} chars)")
    print(f"\nBody sections ({len(d['body'])}):")
    for s in d["body"]:
        print(f"  - {s['title']}  ({len(s['text'])} chars)")

longest = df.loc[df["body_chars"].idxmax(), "pmcid"]
shortest = df.loc[df["body_chars"].idxmin(), "pmcid"]
print("=== LÄNGSTES PAPER ===")
show_paper(longest)
print("\n=== KÜRZESTES PAPER ===")
show_paper(shortest)

=== LÄNGSTES PAPER ===
PMCID:   8453449
PMID:    31085023
DOI:     10.1016/j.hrthm.2019.03.002
Journal: Heart rhythm
Title:   2019 HRS/EHRA/APHRS/LAHRS expert consensus statement on catheter ablation of ventricular arrhythmias
Authors: 38 (first 3: Cronin, Edmond M., Bogun, Frank M., Maury, Philippe)

Abstract sections (1):
  - Abstract  (1678 chars)

Body sections (12):
  - Introduction  (7338 chars)
  - Background  (22271 chars)
  - Clinical Evaluation  (59135 chars)
  - Procedural Planning  (38475 chars)
  - Intraprocedural Patient Care  (37279 chars)
  - Electrophysiological Testing  (4999 chars)
  - Mapping and Imaging Techniques  (41861 chars)
  - Mapping and Ablation  (167602 chars)
  - Postprocedural Care  (46881 chars)
  - Training and Institutional Requirements and Competencies  (11901 chars)
  - Future Directions  (13673 chars)
  - Supplementary Material  (33 chars)

=== KÜRZESTES PAPER ===
PMCID:   10263424
PMID:    37132673
DOI:     10.36660/abc.20230172
Journal: Arquivos 

In [52]:
sample = json.loads((PROCESSED_DIR / f"{longest}.json").read_text())
print(f"--- {sample['body'][0]['title']} ---")
print(sample["body"][0]["text"][:600] + "...")

--- Introduction ---
Section 1Introduction1.1.Document Scope and RationaleThe field of electrophysiology has undergone rapid progress in the last decade, with advances both in our understanding of the genesis of ventricular arrhythmias (VAs) and in the technology used to treat them. In 2009, a joint task force of the European Heart Rhythm Association (EHRA) and the Heart Rhythm Society (HRS), in collaboration with the American College of Cardiology (ACC) and the American Heart Association (AHA), produced an expert consensus document that outlined the state of the field and defined the indications, techniques, and ...
